In [ ]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_openai import ChatOpenAI
import chromadb
from rich.console import Console
from rich.markdown import Markdown

Load the desired pdf file

In [7]:
doc = fitz.open("ACADEMIC_POLICY_MANUAL_14v6.pdf")
pages = [page.get_text() for page in doc]
doc.close()
full_text = "\n\n".join(pages)
print(f"Loaded {len(pages)} pages from PDF")

# Apply Recursive Chunking (the winner from previous lecture)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.split_text(full_text)
print(f"Created {len(recursive_chunks)} chunks using Recursive strategy")
print(f"  - Chunk size target: 500 characters")
print(f"  - Chunk overlap: 50 characters")

Loaded 66 pages from PDF
Created 283 chunks using Recursive strategy
  - Chunk size target: 500 characters
  - Chunk overlap: 50 characters


Load Embedding Model {Qwen or BAAI or miniLLM}

In [8]:
embeddings = HuggingFaceEmbeddings(
    #model_name="BAAI/bge-small-en-v1.5",  # Small, fast, excellent performance
    model_name="Qwen/Qwen3-Embedding-0.6B",  # Larger, more accurate, but slower
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading an already stored embeddings

In [9]:
# Load recursive chunks
client_recursive = chromadb.PersistentClient(path='chroma/IBA_AM_2026/recursive')
vectordb_recursive = Chroma(
    client=client_recursive,
    collection_name="recursive_chunks",
    embedding_function=embeddings
)
# Verify it loaded
print(f"Loaded {vectordb_recursive._collection.count()} chunks from existing index")

/tmp/ipykernel_863/1428927684.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb_recursive = Chroma(


Loaded 283 chunks from existing index


In [10]:
bm25_retriever = BM25Retriever.from_texts(
    recursive_chunks,
    k=3  # Return top 3 by default
)
print(f"BM25 index created with {len(recursive_chunks)} chunks")
print("  - Algorithm: Okapi BM25 (keyword-based ranking)")
print("  - Parameters: k1=1.5, b=0.75 (defaults)")

BM25 index created with 283 chunks
  - Algorithm: Okapi BM25 (keyword-based ranking)
  - Parameters: k1=1.5, b=0.75 (defaults)


Hybrdi Search

In [21]:
# Create semantic retriever
recursive_retriever = vectordb_recursive.as_retriever(
    search_kwargs={"k": 3}
)

# Create ensemble (hybrid) retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, recursive_retriever],
    weights=[0.5, 0.5],  # Equal weight to both methods
    c=3 #constant 1/(c + rank) 
)
print("Hybrid retriever created")
print("  - Combines: BM25 (keyword) + Semantic (vector)")
print("  - Weights: 50% BM25, 50% Semantic")

Hybrid retriever created
  - Combines: BM25 (keyword) + Semantic (vector)
  - Weights: 50% BM25, 50% Semantic


In [19]:
print(f"Number of retrievers: {len(ensemble_retriever.retrievers)}")
print(f"Weights: {ensemble_retriever.weights}")
print(f"RRF constant (c): {ensemble_retriever.c}")  # Default is 60

Number of retrievers: 2
Weights: [0.5, 0.5]
RRF constant (c): 60


Query Question

In [13]:
query = "What is the makeup exam policy for health related reasons?"
print(f"\nQuery: {query}")


Query: What is the makeup exam policy for health related reasons?


BM25 Retriever

In [ ]:
# Method 1: BM25 Only
print("\n" + "-" * 80)
print("METHOD 1: BM25 (KEYWORD SEARCH)")
print("-" * 80)
bm25_results = bm25_retriever.invoke(query) 
for i, doc in enumerate(bm25_results, 1):
    print(f"\nRank {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Length: {len(doc.page_content)} chars")


--------------------------------------------------------------------------------
METHOD 1: BM25 (KEYWORD SEARCH)
--------------------------------------------------------------------------------

Rank 1:
Content: in light of the guidelines given below for cases not covered above. 
• 
Option of re-conducting exams: 
The teacher may develop a makeup exam for the student if possible. The teacher needs to 
ensure ...
Length: 459 chars

Rank 2:
Content: scheduled exam: 
a. A certificate from his/her organization giving details of the official assignment. 
b. Evidence of official travel. 
The student shall be required to appear in the makeup of a term...
Length: 489 chars

Rank 3:
Content: Academic Policy Manual 
 
2014-2015 
IBA Academic sub-committee/ Version Number: 0006 
MS Economics (Evening Program) 
MS program in Economics is designed to provide a solid background in theory, quan...
Length: 482 chars


Semantic Retriever

In [15]:
# Method 2: Semantic Only
print("\n" + "-" * 80)
print("METHOD 2: SEMANTIC SEARCH (VECTOR SIMILARITY)")
print("-" * 80)
recursive_results = vectordb_recursive.similarity_search(query, k=3)
for i, doc in enumerate(recursive_results, 1):
    print(f"\nRank {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Length: {len(doc.page_content)} chars")


--------------------------------------------------------------------------------
METHOD 2: SEMANTIC SEARCH (VECTOR SIMILARITY)
--------------------------------------------------------------------------------

Rank 1:
Content: authenticated by recognized hospitals. The decision will be taken by the Executive 
Committee. The Committee’s decision in this regard shall be final. This facility shall, 
however, be allowed only fo...
Length: 479 chars

Rank 2:
Content: • 
Makeup Final exam: 
If the student has missed the final exam, a simple exam or assignment is not recommended 
as the student has not been tested on a large portion of the syllabus. 
 
Source: IBA P...
Length: 482 chars

Rank 3:
Content: scheduled exam: 
a. A certificate from his/her organization giving details of the official assignment. 
b. Evidence of official travel. 
The student shall be required to appear in the makeup of a term...
Length: 489 chars


Hybrid Retriever

In [22]:
hybrid_results = ensemble_retriever.invoke(query)[:3]
for i, doc in enumerate(hybrid_results, 1):
    print(f"\nRank {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Length: {len(doc.page_content)} chars")


Rank 1:
Content: scheduled exam: 
a. A certificate from his/her organization giving details of the official assignment. 
b. Evidence of official travel. 
The student shall be required to appear in the makeup of a term...
Length: 489 chars

Rank 2:
Content: in light of the guidelines given below for cases not covered above. 
• 
Option of re-conducting exams: 
The teacher may develop a makeup exam for the student if possible. The teacher needs to 
ensure ...
Length: 459 chars

Rank 3:
Content: authenticated by recognized hospitals. The decision will be taken by the Executive 
Committee. The Committee’s decision in this regard shall be final. This facility shall, 
however, be allowed only fo...
Length: 479 chars


Connecting with LM Studio

In [33]:
# Example: reuse your existing OpenAI setup
from openai import OpenAI

llm = ChatOpenAI(
    base_url="http://172.19.64.1:1234/v1",
    api_key="lm-studio",
    #model="google/gemma-3-12b",
    model="tiny-aya-global",
    temperature=0.7
)

Hybrid Retriever

In [34]:
# Prepare context from hybrid results (best method)
context = "\n\n---\n\n".join([
    f"Excerpt {i+1}:\n{doc.page_content[:800]}" 
    for i, doc in enumerate(hybrid_results)
])

prompt = f"""
Based on the following information:\n\n
{context}
Please provide a detailed answer to the question: {query}.
Your answer should integrate the essence of all three responses, providing a unified answer that leverages the \
diverse perspectives or data points provided by three responses. \
If the responses are irrelevant to the question then respond by saying that I couldn't find a good response to your query in the database. 
"""



#print(completion.choices[0].message)

#import textwrap
#print(textwrap.fill(completion.choices[0].message.content, 80))


In [ ]:
response = llm.invoke(prompt)

console = Console()

md = Markdown(response.content)
console.print(md)

Based on the provided information and combining the key elements from Excerpts 1, 2, and 3, here's a unified makeup
exam policy for health-related reasons:                                                                            

Make-up Exam Policy for Health-Related Reasons:                                                                    

 1 Medical Make-up Exam: A student can request a make-up exam due to medical grounds in extremely serious cases,   
   but this option is strictly limited to one of the two term examinations in a semester (as per Excerpt 3). This  
   means that:                                                                                                     
    • Students can only take one medical-related makeup exam during a semester.                                    
    • Make-up exams are not available for semester finals or any other term examinations.                          
 2 Evidence Requirements: To request a medical make-up exam, students must provide authenticated proof from        
   recognized hospitals, as mentioned in Excerpt 3.                                                                
 3 Decision-Making Process: The decision to approve a medical make-up exam will be made by the Executive Committee,
   who will have final say on such matters (Excerpt 3).                                                            
 4 Examination Fee: If a student chooses to take a makeup exam due to health reasons, they are required to pay an  
   examination fee of Rs. 2000/- for one subject (Excerpt 1).                                                      
 5 Re-conducting Exams: In cases where the missed exam was difficult and not unfair, teachers can organize a       
   make-up exam as an option (Excerpt 2). However:                                                                 
    • Make-up exams are limited to one missed exam per student.                                                    
    • Teachers must ensure that the makeup exam does not provide an unfair advantage to the student.               

This policy aims to provide flexibility for students facing health issues while maintaining academic integrity and 
fairness. It allows for medical make-ups within specific constraints, ensuring that these opportunities are not    
abused or misused.<|END_RESPONSE|>